In [1]:
from pathlib import Path

import torch
import torchaudio
from huggingface_hub import hf_hub_download

from moshi.models import LMGen, loaders

# 1. Download weights from Hugging Face
mimi_ckpt = hf_hub_download(loaders.DEFAULT_REPO, loaders.MIMI_NAME)
moshi_ckpt = hf_hub_download(loaders.DEFAULT_REPO, loaders.MOSHI_NAME)

/home/saeki/Documents/moshi/moshi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 2. Initialize Mimi (audio codec) and Moshi LM
device = "cuda:0" if torch.cuda.is_available() else "cpu"
mimi = loaders.get_mimi(mimi_ckpt, device=device)
mimi.set_num_codebooks(8)  # up to 32 for full Mimi; 8 is default for Moshi

moshi_lm = loaders.get_moshi_lm(moshi_ckpt, device=device)
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)

print("Moshi LM architecture:")
print(moshi_lm)

Moshi LM architecture:
LMModel(
  (emb): ModuleList(
    (0-15): 16 x ScaledEmbedding(2049, 4096)
  )
  (text_emb): ScaledEmbedding(32001, 4096)
  (text_linear): Linear(in_features=4096, out_features=32000, bias=False)
  (transformer): StreamingTransformer(
    (rope): RotaryEmbedding()
    (layers): ModuleList(
      (0-31): 32 x StreamingTransformerLayer(
        (self_attn): StreamingMultiheadAttention(
          (rope): RotaryEmbedding()
          (out_projs): ModuleList(
            (0): Linear(in_features=4096, out_features=4096, bias=False)
          )
          (in_projs): ModuleList(
            (0): Linear(in_features=4096, out_features=12288, bias=False)
          )
        )
        (norm1): RMSNorm()
        (norm2): RMSNorm()
        (gating): ActivationGating(
          (linear_in): Linear(in_features=4096, out_features=22528, bias=False)
          (linear_out): Linear(in_features=11264, out_features=4096, bias=False)
        )
        (layer_scale_1): Identity()
       

In [8]:
# 3. Load and (optionally) resample your input audio
audio_path = Path("input.wav")
wav, sr = torchaudio.load(audio_path)  # [1, T], any sr
if sr != 24000:
    wav = torchaudio.functional.resample(wav, sr, 24000)
wav = wav.unsqueeze(0)  # [B=1, C=1, T]
wav = wav.to(device)

# 4. Encode entire signal into Mimi codes (non-streaming demo)
with torch.no_grad():
    full_codes = mimi.encode(wav)  # [1, K=8, T_frames]

print(full_codes)
print(full_codes.shape)

tensor([[[1049,  127,  319,  ...,  598, 1833,  855],
         [1597,  655, 1783,  ..., 1783,  985, 2038],
         [1626, 1626,  373,  ..., 1496,  642,  869],
         ...,
         [1572, 1040,  599,  ..., 1707, 2019,  457],
         [1045,  930, 2041,  ..., 1748,   34,  557],
         [1904,  189, 1843,  ..., 1549,  124, 1842]]], device='cuda:0')
torch.Size([1, 8, 250])


In [ ]:
# 5. Stream-based inference: feed Mimi codes into Moshi, decode on the fly
frame_size = mimi.frame_size
out_chunks = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    # First, get the raw Mimi codes broken into frames:
    for offset in range(0, full_codes.shape[-1], 1):  # one code-frame at a time
        code_frame = full_codes[:, :, offset : offset + 1].to(device)
        # Moshi generates audio+text tokens; here we step one code-frame
        tokens = lm_gen.step(code_frame)
        if tokens is not None:
            # tokens[:, 1:] are Mimi audio codes (first slot is text)
            audio_codes = tokens[:, 1:]
            chunk = mimi.decode(audio_codes)
            out_chunks.append(chunk.cpu())

tensor([[[   3],
         [1049],
         [1700],
         [1626],
         [ 546],
         [ 306],
         [1443],
         [1871],
         [2008]]], device='cuda:0')


RuntimeError: No active exception to reraise

In [ ]:
# 6. Concatenate output chunks and save
output_wav = torch.cat(out_chunks, dim=-1)  # [1, 1, T_out]
torchaudio.save("moshi_output.wav", output_wav.squeeze(0), 24000)
print("Saved generated audio to moshi_output.wav")